In [ ]:
# Step 1: Import All Required SeQUeNCe Modules

import numpy as np
from numpy import multiply
from scipy.special import j0
from scipy.stats import poisson

from sequence.kernel.timeline import Timeline
from sequence.kernel.entity import Entity
from sequence.kernel.event import Event
from sequence.kernel.process import Process

from sequence.topology.node import QKDNode

from sequence.components.optical_channel import QuantumChannel, ClassicalChannel
from sequence.components.light_source import LightSource
from sequence.components.photon import Photon, polarization
from sequence.components.detector import QSDetectorPolarization, Detector

from sequence.qkd.BB84 import BB84, BB84MsgType, pair_bb84_protocols
from sequence.message import Message
from sequence.protocol import StackProtocol
from sequence.constants import SPEED_OF_LIGHT

from sequence.utils import log

import plotly.graph_objects as go

print("BB84MsgType values:", list(BB84MsgType))
print("SPEED_OF_LIGHT:", SPEED_OF_LIGHT)
print("All imports OK")

In [ ]:

# ================= Step 2: Define Parameters =================
import numpy as np
from numpy import multiply
from scipy.special import j0
from sequence.components.optical_channel import QuantumChannel
from sequence.components.light_source import LightSource
from sequence.components.photon import Photon, polarization
from sequence.kernel.event import Event
from sequence.kernel.process import Process
from sequence.utils import log

MIMO_configs = [4, 8, 12]                     # editable
distances_km_graph = np.arange(2, 10 + 1, 2) # editable

mu_s, mu_1, mu_2 = 0.5, 0.1, 0.001    # decoy-state mean photon numbers (paper Sec. V)
decoy_probs = (0.5, 0.25, 0.25)       # selection probability for (mu_s, mu_1, mu_2)

wavelength = 1550e-9
w = 0.035
ar = 0.20
delta = 0.43e-3
Cn2 = 1e-15
theta_p = 1e-6
eta_d = 0.12
Y0 = 1.6e-5
e_det = 0.015
q = 0.5
g_val = 1.03
e0 = 0.5

N_BITS = 20
#FAST_LIGHT_SPEED = 1e10   # near-instant propagation (avoids BB84 timing mismatch)

print("Parameters set:")
print("MIMO_configs:", MIMO_configs)
print("distances_km_graph:", distances_km_graph)
print("mu_s, mu_1, mu_2:", mu_s, mu_1, mu_2, "| probs:", decoy_probs)
print("Y0:", Y0, "| e_det:", e_det, "| eta_d:", eta_d)
print("N_BITS:", N_BITS)


# ================= Step 3: Custom MIMO-FSO Quantum Channel Class (FULL, latest version) =================

class MIMOFSOChannel(QuantumChannel):
    """Quantum channel replacing standard fiber loss with MIMO-FSO transmissivity
    (beam spreading, pointing error, beam wandering, turbulence, atmospheric
    attenuation), following the paper's model. Also intercepts photons for
    Eve's PNS attack if an EveEntity is attached (see Step 5).
    """

    def __init__(self, name, timeline, distance, NT, NR,
                 wavelength=1550e-9, w=0.035, ar=0.20,
                 delta=0.43e-3, Cn2=1e-15, theta_p=1e-6, eta_d=0.12,
                 polarization_fidelity=1.0, light_speed=2e-4, frequency=8e7,
                 eve=None):

        super().__init__(name, timeline, attenuation=0, distance=distance,
                          polarization_fidelity=polarization_fidelity,
                          light_speed=light_speed, frequency=frequency)

        self.NT, self.NR = NT, NR
        self.wavelength, self.w, self.ar = wavelength, w, ar
        self.delta, self.Cn2, self.theta_p, self.eta_d = delta, Cn2, theta_p, eta_d
        self.k = 2 * np.pi / wavelength

        self.betas = None
        self.Ti_all = None
        self.eve = eve   # EveEntity instance, or None (no attack)
        self.transmission_log = []   # (qubit_name, outcome: 'blocked'/'transmitted'/'lost')

    def init(self) -> None:
        self.delay = round(self.distance / self.light_speed)
        z = self.distance
        aperture_area = np.pi * self.ar ** 2

        H, _, _ = self._build_H_matrix(z, aperture_area)
        self.betas = np.linalg.svd(H, compute_uv=False)

        Ta = 10 ** (-self.delta * z / 10)

        d = self.ar * np.sqrt(self.k / z)
        chi2 = 1.23 * self.Cn2 * (self.k ** (7/6)) * (z ** (11/6))
        xi1 = 0.49*chi2 / (1 + 0.18*d**2 + 0.56*chi2**(12/5))**(7/6)
        xi2 = 0.51*chi2 / (1 + 0.9*d**2 + 0.62*(d**2)*chi2**(12/5))**(5/6)
        sigma2_turb = np.exp(xi1 + xi2) - 1
        Tt = np.exp(-0.5 * sigma2_turb)

        self.Ti_all = self.eta_d * Ta * Tt * self.betas
        self.loss = 1 - np.max(self.Ti_all)

        print(f"[{self.name}] init(): NT={self.NT} NR={self.NR} distance={z}m")
        print(f"[{self.name}] betas: {self.betas}")
        print(f"[{self.name}] Ti_all: {self.Ti_all}")
        print(f"[{self.name}] loss: {self.loss}")

    def calc_Tn(self, n, sub_channel_idx=None):
        """Photon-number-dependent transmittance Tn,i (Eq. 12)."""
        Ti = np.max(self.Ti_all) if sub_channel_idx is None else self.Ti_all[sub_channel_idx]
        return 1 - (1 - Ti) ** n

    def transmit(self, qubit: "Photon", source) -> None:
        """Overrides QuantumChannel.transmit(): inserts Eve's PNS attack (if
        attached), then explicitly shows the channel-loss pass/fail decision
        for each photon (Step 21: Photon Transmission Log).
        """
        if self.eve is not None:
            forward = self.eve.intercept(qubit)
            if not forward:
                print(f"[{self.name}] Photon '{qubit.name}' BLOCKED by Eve (stolen copy)")
                self.transmission_log.append((qubit.name, "blocked_by_eve"))
                return

        assert self.delay >= 0 and self.loss <= 1
        assert source == self.sender

        survives = (self.sender.get_generator().random() > self.loss) or qubit.is_null

        if survives:
            print(f"[{self.name}] Photon '{qubit.name}' TRANSMITTED (survived loss={self.loss:.4f}) -> Bob in {self.delay} ps")
            self.transmission_log.append((qubit.name, "transmitted"))
            if qubit.is_null:
                qubit.add_loss(self.loss)
            if (qubit.encoding_type["name"] == "polarization"
                    and self.sender.get_generator().random() > self.polarization_fidelity):
                qubit.random_noise(self.get_generator())
            future_time = self.timeline.now() + self.delay
            process = Process(self.receiver, "receive_qubit", [source.name, qubit])
            event = Event(future_time, process)
            self.timeline.schedule(event)
        else:
            print(f"[{self.name}] Photon '{qubit.name}' LOST in channel (loss={self.loss:.4f})")
            self.transmission_log.append((qubit.name, "lost_in_channel"))

    def _build_H_matrix(self, z, aperture_area, n_points=60):
        R0 = np.sqrt(self.NT) * self.w
        tx_positions = self._square_grid_positions(self.NT, R0)
        rx_positions = self._square_grid_positions(self.NR, R0)
        r_points = np.linspace(0, R0, n_points)
        E_values = np.sqrt(2/(np.pi*self.w**2)) * np.exp(-r_points**2/self.w**2)
        rho_max = np.sin(self.wavelength/(np.pi*self.w))/self.wavelength
        rho_points = np.linspace(0, rho_max, n_points)
        F_values = np.zeros(n_points)
        for k_idx, rho in enumerate(rho_points):
            integrand = r_points * E_values * j0(2*np.pi*r_points*rho)
            F_values[k_idx] = 2*np.pi*np.trapezoid(integrand, r_points)
        norm_val = np.trapezoid(r_points * E_values**2, r_points)
        denom = np.sqrt(2*np.pi*norm_val)
        H = np.zeros((self.NR, self.NT), dtype=complex)
        for i in range(self.NR):
            for j in range(self.NT):
                dist = np.linalg.norm(rx_positions[i] - tx_positions[j])
                G_val = self._G_distance(dist, rho_points, F_values, z)
                H[i, j] = (aperture_area * G_val) / denom
        return H, tx_positions, rx_positions

    def _G_distance(self, r, rho_points, F_values, z):
        arg = self.k**2 - (2*np.pi*rho_points)**2
        phase = np.sqrt(np.maximum(arg, 0.0)) * z
        bessel = j0(2*np.pi*r*rho_points)
        real_integrand = rho_points * F_values * bessel * np.cos(phase)
        imag_integrand = rho_points * F_values * bessel * np.sin(phase)
        real_part = 2*np.pi*np.trapezoid(real_integrand, rho_points)
        imag_part = 2*np.pi*np.trapezoid(imag_integrand, rho_points)
        return real_part + 1j*imag_part

    @staticmethod
    def _square_grid_positions(N, R0):
        side = int(np.ceil(np.sqrt(N)))
        spacing = (2*R0)/(side+1)
        positions = []
        for ix in range(side):
            for iy in range(side):
                x = -R0 + spacing*(ix+1)
                y = -R0 + spacing*(iy+1)
                positions.append((x, y))
        return np.array(positions[:N])



# ================= Step 4: Custom Decoy-State WCP Light Source Class =================
class DecoyWCPSource(LightSource):
    """Weak Coherent Pulse source with two-decoy-state method (Eq. 11).
    For each pulse, randomly selects mu_i from {mu_s, mu_1, mu_2}, samples
    photon number ~ Poisson(mu_i), and tags each emitted photon with its
    pulse index and total photon count (needed for Eve's PNS logic).
    """

    def __init__(self, name, timeline, mu_s=0.5, mu_1=0.1, mu_2=0.001,
                 probs=(0.5, 0.25, 0.25), frequency=8e7, wavelength=1550,
                 bandwidth=0, encoding_type=None, phase_error=0):

        if encoding_type is None:
            encoding_type = polarization

        super().__init__(name, timeline, frequency=frequency, wavelength=wavelength,
                          bandwidth=bandwidth, mean_photon_num=mu_s,
                          encoding_type=encoding_type, phase_error=phase_error)

        self.mu_s, self.mu_1, self.mu_2 = mu_s, mu_1, mu_2
        self.probs = probs
        self.mu_sequence = []       # mu chosen per pulse
        self.photon_count_log = []  # (pulse_index, mu, n_photons)
        self.full_log = []          # (pulse_index, bit, basis, mu, n_photons)

        # reverse lookup: quantum state -> (basis, bit), for logging purposes only
        self._state_to_bit_basis = {}
        for basis_idx in range(2):
            for bit_idx in range(2):
                state = self.encoding_type["bases"][basis_idx][bit_idx]
                self._state_to_bit_basis[state] = (basis_idx, bit_idx)

    def emit(self, state_list) -> None:
        print(f"[{self.name}] emit(): generating {len(state_list)} decoy-state WCP pulses")

        time = self.timeline.now()
        period = int(round(1e12 / self.frequency))
        mu_states = [self.mu_s, self.mu_1, self.mu_2]

        for i, state in enumerate(state_list):
            mu_i = self.get_generator().choice(mu_states, p=self.probs)
            self.mu_sequence.append(mu_i)

            num_photons = self.get_generator().poisson(mu_i)
            self.photon_count_log.append((i, mu_i, num_photons))

            basis_idx, bit_idx = self._state_to_bit_basis.get(
                tuple(state) if isinstance(state, list) else state, (None, None))
            self.full_log.append((i, bit_idx, basis_idx, mu_i, num_photons))
            print(f"  Pulse {i}: bit={bit_idx} basis={basis_idx} mu={mu_i:.3f} -> n_photons={num_photons}")

            if self.get_generator().random() < self.phase_error:
                state = multiply([1, -1], state)

            for copy_idx in range(num_photons):
                wl = self.linewidth * self.get_generator().standard_normal() + self.wavelength
                new_photon = Photon(str(i), self.timeline, wavelength=wl,
                                     location=self.owner, encoding_type=self.encoding_type,
                                     quantum_state=state)
                new_photon.pulse_index = i
                new_photon.n_total = num_photons
                new_photon.copy_index = copy_idx

                process = Process(self._receivers[0], "get", [new_photon])
                event = Event(time, process)
                self.timeline.schedule(event)
                self.photon_counter += 1

            time += period




# ================= Step 5: Custom Eve Entity (PNS Attack) =================
class EveEntity:
    def __init__(self, name):
        self.name = name
        self.decided_pulses = {}
        self.stored_photons = []
        self.attack_log = []

    def intercept(self, photon: "Photon") -> bool:
        pulse_idx = getattr(photon, "pulse_index", None)
        n_total = getattr(photon, "n_total", 1)
        copy_idx = getattr(photon, "copy_index", 0)

        if pulse_idx is None:
            return True

        if n_total <= 1:
            action = "forward (single photon, cannot split)"
            self.attack_log.append((pulse_idx, n_total, copy_idx, action))
            print(f"[{self.name}] Pulse {pulse_idx} (n={n_total}): {action}")
            return True

        if pulse_idx not in self.decided_pulses:
            self.decided_pulses[pulse_idx] = True
            self.stored_photons.append(pulse_idx)
            action = f"STOLEN (copy {copy_idx}) -> stored in quantum memory"
            self.attack_log.append((pulse_idx, n_total, copy_idx, action))
            print(f"[{self.name}] Pulse {pulse_idx} (n={n_total}): {action}")
            return False
        else:
            action = f"forwarded (copy {copy_idx}, remaining photon)"
            self.attack_log.append((pulse_idx, n_total, copy_idx, action))
            print(f"[{self.name}] Pulse {pulse_idx} (n={n_total}): {action}")
            return True


# ================= Step 6: Custom Detector Configuration (Threshold Behavior) =================
# SeQUeNCe's built-in Detector already supports dark_count natively (= Y0).
# e_det (misalignment error) is modeled via the channel's polarization_fidelity.
# No custom class needed -- QSDetectorPolarization (built-in) configured later
# with dark_count=Y0 on each internal detector.

print()
print("Step 2-6 complete: parameters set, MIMOFSOChannel, DecoyWCPSource, EveEntity defined.")
print("Detector: using built-in QSDetectorPolarization with dark_count=Y0 (configured later).")


In [ ]:
from sequence.kernel.timeline import Timeline
from sequence.topology.node import QKDNode

# Step 7: Create Timeline
tl = Timeline(10e12)

# Step 8: Create Alice Node
alice = QKDNode("Alice", tl, stack_size=0)

# Step 9: Create Bob Node
bob = QKDNode("Bob", tl, stack_size=0)

print("Timeline:", tl, "| now():", tl.now())
print("Alice node:", alice.name)
print("Bob node:", bob.name)

In [ ]:
# ================= Step 10: Attach MIMO-FSO Quantum Channels (Alice<->Bob) with Eve Interception =================

NT = MIMO_configs[0]                  # will be overwritten by the loop in Step 30
distance_km = distances_km_graph[0]   # will be overwritten by the loop in Step 30
distance_m = distance_km * 1000

# Step 5 (instantiate): Eve Entity, attached only on the Alice->Bob direction
eve = EveEntity("Eve")

qc_a2b = MIMOFSOChannel("qc_a2b", tl, distance=distance_m, NT=NT, NR=NT,
                         polarization_fidelity=1 - e_det, eve=eve)
qc_a2b.set_ends(alice, bob.name)

qc_b2a = MIMOFSOChannel("qc_b2a", tl, distance=distance_m, NT=NT, NR=NT,
                         polarization_fidelity=1 - e_det)
qc_b2a.set_ends(bob, alice.name)

print("NT = NR =", NT, "| distance =", distance_km, "km")
print("qc_a2b:", qc_a2b.name, "| Eve attached:", qc_a2b.eve is not None)
print("qc_b2a:", qc_b2a.name, "| Eve attached:", qc_b2a.eve is not None)

In [ ]:
# ================= Step 11: Attach Classical Channels (Alice<->Bob) =================
from sequence.components.optical_channel import ClassicalChannel

cc_a2b = ClassicalChannel("cc_a2b", tl, distance=distance_m)
cc_a2b.set_ends(alice, bob.name)

cc_b2a = ClassicalChannel("cc_b2a", tl, distance=distance_m)
cc_b2a.set_ends(bob, alice.name)

print("cc_a2b:", cc_a2b.name, "| distance:", cc_a2b.distance)
print("cc_b2a:", cc_b2a.name, "| distance:", cc_b2a.distance)

In [ ]:
# ================= Step 12: Attach Light Source to Alice =================

alice_source = DecoyWCPSource("Alice.wcp_source", tl, mu_s=mu_s, mu_1=mu_1, mu_2=mu_2, probs=decoy_probs)
alice.add_component(alice_source)
alice_source.add_receiver(alice)   # emitted photons route through alice -> qchannel
alice.destination = "Bob"          # tells QKDNode.get() where to send photons

print("Alice components:", list(alice.components.keys()))
print("Alice destination:", alice.destination)
print("Light source mu_s, mu_1, mu_2:", alice_source.mu_s, alice_source.mu_1, alice_source.mu_2)
print("Light source probs:", alice_source.probs)

In [ ]:
# ================= Step 13: Attach Detector to Bob =================
from sequence.components.detector import QSDetectorPolarization

bob_qsdetector = QSDetectorPolarization("Bob.qsdetector_custom", tl)
for d in bob_qsdetector.detectors:
    d.dark_count = Y0   # background/dark count rate (paper Sec. V)

bob.add_component(bob_qsdetector)
for d in bob_qsdetector.detectors:
    bob.add_component(d)
bob.add_component(bob_qsdetector.splitter)
bob.set_first_component(bob_qsdetector.name)   # incoming photons routed here

print("Bob components:", list(bob.components.keys()))
print("Bob first_component_name:", bob.first_component_name)
print("Detector 0 dark_count:", bob_qsdetector.detectors[0].dark_count)
print("Detector 1 dark_count:", bob_qsdetector.detectors[1].dark_count)

In [ ]:
detection_log = []   # (detector_label, photon_name, outcome)

def make_logging_get(detector, label):
    original_record = detector.record_detection
    def logging_get(photon=None, **kwargs):
        detector.photon_counter += 1
        if photon and photon.encoding_type["name"] == "single_atom":
            key = photon.quantum_state
            res = detector.timeline.quantum_manager.run_circuit(
                type(detector)._meas_circuit, [key], detector.get_generator().random())
            if not res[key]:
                detection_log.append((label, photon.name if photon else "?", "NO-CLICK (measured |0>)"))
                return
        if detector.get_generator().random() < detector.efficiency:
            detection_log.append((label, photon.name if photon else "?", "CLICK"))
            original_record()
        else:
            detection_log.append((label, photon.name if photon else "?", "NO-CLICK (detector inefficiency)"))
    return logging_get

bob_qsdetector.detectors[0].get = make_logging_get(bob_qsdetector.detectors[0], "Bob.detector0")
bob_qsdetector.detectors[1].get = make_logging_get(bob_qsdetector.detectors[1], "Bob.detector1")

print("Bob's detectors patched for click/no-click logging (stored in detection_log).")

In [ ]:
# ================= Step 14: Create BB84 Protocol Instances =================
from sequence.qkd.BB84 import BB84

bb84_alice = BB84(alice, "bb84_alice", "Alice.wcp_source", "Bob.qsdetector_custom")
bb84_bob = BB84(bob, "bb84_bob", "Alice.wcp_source", "Bob.qsdetector_custom")

alice.protocols.append(bb84_alice)
bob.protocols.append(bb84_bob)

print("bb84_alice:", bb84_alice.name, "| role:", bb84_alice.role)
print("bb84_bob:", bb84_bob.name, "| role:", bb84_bob.role)

In [ ]:
# ================= Step 15: Pair BB84 Protocols =================
from sequence.qkd.BB84 import pair_bb84_protocols

pair_bb84_protocols(bb84_alice, bb84_bob)   # sets roles: alice=0 (sender), bob=1 (receiver)

print("Alice paired with:", bb84_alice.another.name, "| role:", bb84_alice.role)
print("Bob paired with:  ", bb84_bob.another.name, "| role:", bb84_bob.role)

In [ ]:
from sequence.qkd.BB84 import BB84MsgType

comparison_log = []   # (index, alice_basis, bob_basis, bob_bit, status)
sifting_log = []      # (total_pulses, kept_indices, sifted_bits, kept_count, discarded_count)

def make_logging_received_message(protocol, label):
    original = protocol.received_message
    def logging_received_message(src, msg):
        if msg.msg_type is BB84MsgType.BASIS_LIST and len(protocol.basis_lists) > 0:
            basis_list_alice = msg.bases
            basis_list = protocol.basis_lists[0]
            bits = protocol.bit_lists[0]
            kept_indices = []
            for i, b in enumerate(basis_list_alice):
                bit_val = bits[i]
                own_basis = basis_list[i]
                if bit_val == -1:
                    status = "NO DETECTION (no click)"
                elif own_basis == b:
                    status = f"BASIS MATCH -> bit={bit_val} kept"
                    kept_indices.append(i)
                else:
                    status = "BASIS MISMATCH -> discarded"
                comparison_log.append((i, b, own_basis, bit_val, status))

            sifting_log.append((
                len(basis_list_alice), kept_indices,
                [bits[i] for i in kept_indices],
                len(kept_indices), len(basis_list_alice) - len(kept_indices)
            ))
        return original(src, msg)
    return logging_received_message

bb84_bob.received_message = make_logging_received_message(bb84_bob, "bb84_bob")
print("Bob's received_message patched (comparison_log, sifting_log will be filled during run).")

In [ ]:
# ================= Step 16: Initialize Timeline =================
tl.init()

print("Timeline initialized successfully")

In [ ]:
# ================= Step 17: Show Channel Parameters After Initialization =================

print(f"===== Channel Parameters (NT=NR={NT}, distance={distance_km} km) =====")
print("Singular values (betas):", qc_a2b.betas)
print("Effective transmissivity per sub-channel (Ti_all):", qc_a2b.Ti_all)
print("Number of sub-channels (rH):", len(qc_a2b.Ti_all))
print("Channel loss (used internally by transmit()):", qc_a2b.loss)
print("Strongest sub-channel Ti:", np.max(qc_a2b.Ti_all))

In [ ]:
# ================= Step 18: Set Key Length & Push Protocol Request =================
# (proper fix: adjust mean_photon_num so SeQUeNCe's own light_time formula
# comes out large enough to exceed round-trip delay, before pushing)

qc_delay = qc_a2b.delay
cc_delay = cc_a2b.delay
round_trip_ps = qc_delay + 3 * cc_delay   # 1 qc leg + 3 cc legs (RECEIVED_QUBITS, BASIS_LIST, MATCHING_INDICES)

MARGIN = 5
light_time_target_s = (round_trip_ps * MARGIN) / 1e12
mean_photon_num_needed = N_BITS / (alice_source.frequency * light_time_target_s)
alice_source.mean_photon_num = mean_photon_num_needed   # only affects SeQUeNCe's internal timing estimate

run_time_ps = light_time_target_s * 1e12 + 3 * cc_delay + round_trip_ps

print("qc_delay:", qc_delay, "ps | cc_delay:", cc_delay, "ps")
print("round_trip_ps:", round_trip_ps)
print("light_time_target_s:", light_time_target_s)
print("mean_photon_num_needed (timing-only):", mean_photon_num_needed)
print("run_time_ps:", run_time_ps)

bb84_alice.push(length=N_BITS, key_num=1, run_time=run_time_ps)

print("Alice working:", bb84_alice.working, "| Bob working:", bb84_bob.working)
print("Requested key length:", N_BITS)

In [ ]:
# ================= Step 19: Run Timeline (Execute Full Protocol) =================
# Steps 20-23 (photon generation, transmission, Eve's PNS attack, forwarding)
# print automatically during this call, via DecoyWCPSource.emit() and
# EveEntity.intercept(), since those print statements are built into the classes.

tl.run()

print()
print("Timeline run complete. Final simulation time:", tl.now())

In [ ]:
# ================= Step 22: Display Eve's PNS Attack Log (summary) =================

print(f"===== Eve's PNS Attack Summary =====")
print("Total pulses Eve intercepted:", len(eve.attack_log))
print("Total pulses Eve stole from (n>1):", len(eve.stored_photons))
print()
print("Sample of attack_log (pulse_idx, n_total, copy_idx, action):")
for entry in eve.attack_log[:10]:
    print(" ", entry)
print("...")
for entry in eve.attack_log[-5:]:
    print(" ", entry)

n_single_photon_pulses = sum(1 for e in eve.attack_log if e[1] <= 1)
n_multi_photon_pulses = sum(1 for e in eve.attack_log if e[1] > 1)
print()
print("Single-photon pulses (Eve could not attack):", n_single_photon_pulses)
print("Multi-photon pulses (Eve stole from):", n_multi_photon_pulses)

In [ ]:
# ================= Step 23: Display Photon Forwarding Log (Eve -> Bob) =================

print("===== Photon Forwarding Log (Eve's decision -> Channel outcome) =====")
print("Total photon transmission attempts logged:", len(qc_a2b.transmission_log))
print()
print("Sample (first 10):")
for name, outcome in qc_a2b.transmission_log[:10]:
    print(f"  Photon '{name}': {outcome}")

blocked = sum(1 for _, o in qc_a2b.transmission_log if o == "blocked_by_eve")
transmitted = sum(1 for _, o in qc_a2b.transmission_log if o == "transmitted")
lost = sum(1 for _, o in qc_a2b.transmission_log if o == "lost_in_channel")

print()
print("Blocked by Eve:", blocked)
print("Transmitted (reached Bob):", transmitted)
print("Lost in channel:", lost)
print("Total:", blocked + transmitted + lost)

In [ ]:
print("===== Bob's Reception & Detection Log =====")
print("Total detection attempts logged:", len(detection_log))
print()
print("Sample (first 10):")
for label, photon_name, outcome in detection_log[:10]:
    print(f"  [{label}] Photon '{photon_name}': {outcome}")

clicks = sum(1 for _, _, o in detection_log if o == "CLICK")
no_clicks = len(detection_log) - clicks

print()
print("Total CLICKs:", clicks)
print("Total NO-CLICKs:", no_clicks)

In [ ]:
print("===== Measurement & Basis Comparison (sample) =====")
print("Total comparisons logged:", len(comparison_log))
for i, alice_basis, bob_basis, bob_bit, status in comparison_log[:10]:
    print(f"  index {i}: Alice_basis={alice_basis} Bob_basis={bob_basis} Bob_bit={bob_bit} -> {status}")

print()
print("===== Sifting Summary =====")
for batch_idx, (total, kept_indices, sifted_bits, kept_count, discarded_count) in enumerate(sifting_log):
    print(f"Batch {batch_idx}: total={total} | kept={kept_count} | discarded={discarded_count}")
    print(f"  Sifted (kept) indices (first 20): {kept_indices[:20]}")
    print(f"  Sifted key bits (first 20): {sifted_bits[:20]}")

In [ ]:

# ================= Step 27: Display Final Key (Alice vs Bob, Match Check) =================
# Paste this as a NEW cell right after Step 19 (tl.run())

print("===== Final Key =====")
print("Alice key (int):", bb84_alice.key)
print("Bob key (int):  ", bb84_bob.key)

alice_bin = bin(bb84_alice.key)[2:].zfill(N_BITS)
bob_bin = bin(bb84_bob.key)[2:].zfill(N_BITS)
print("Alice key (bin):", alice_bin)
print("Bob key (bin):  ", bob_bin)

key_match = bb84_alice.key == bb84_bob.key
print()
print("Keys match:", key_match)

if not key_match:
    mismatches = sum(1 for a, b in zip(alice_bin, bob_bin) if a != b)
    print("Bit mismatches:", mismatches, "out of", len(alice_bin))
    print("Simulated raw QBER (from key comparison):", mismatches / len(alice_bin))

In [ ]:
# ================= Step 28: Compute QBER (Paper Formula, from Simulated Channel) =================

def calc_Q(mu, Ti, Y0):
    return Y0 + (1 - Y0) * (1 - np.exp(-mu * Ti))

def calc_E(mu, Ti, Y0, e0, e_det):
    Q = calc_Q(mu, Ti, Y0)
    return (e0*Y0 + e_det*(1 - np.exp(-mu*Ti))) / Q

def calc_Y1_2decoy(mu_s, mu_1, mu_2, Ti, Y0):
    Q_mu1 = calc_Q(mu_1, Ti, Y0)
    Q_mu2 = calc_Q(mu_2, Ti, Y0)
    Q_mus = calc_Q(mu_s, Ti, Y0)
    Y0_L = max((mu_1*Q_mu2*np.exp(mu_2) - mu_2*Q_mu1*np.exp(mu_1)) / (mu_1 - mu_2), 0)
    factor = 1 / ((mu_s - mu_1 - mu_2) * mu_s * (mu_1 - mu_2))
    bracket = mu_s**2 * (Q_mu1*np.exp(mu_1) - Q_mu2*np.exp(mu_2)) - (mu_1**2 - mu_2**2) * (Q_mus*np.exp(mu_s) - Y0_L)
    return factor * bracket

def calc_Q1_2decoy(Y1_2decoy, mu_s):
    return Y1_2decoy * mu_s * np.exp(-mu_s)

def calc_e1_2decoy(mu_1, mu_2, Ti, Y0, e0, e_det, Y1_2decoy):
    Q_mu1 = calc_Q(mu_1, Ti, Y0)
    Q_mu2 = calc_Q(mu_2, Ti, Y0)
    E_mu1 = calc_E(mu_1, Ti, Y0, e0, e_det)
    E_mu2 = calc_E(mu_2, Ti, Y0, e0, e_det)
    numerator = E_mu1*Q_mu1*np.exp(mu_1) - E_mu2*Q_mu2*np.exp(mu_2)
    denominator = (mu_1 - mu_2) * Y1_2decoy
    return numerator / denominator

# Use the actual simulated channel's Ti_all (from qc_a2b, computed during Step 16 tl.init())
Ti_all_sim = qc_a2b.Ti_all

e1_all, Q1_all = [], []
for Ti in Ti_all_sim:
    Y1_2decoy = calc_Y1_2decoy(mu_s, mu_1, mu_2, Ti, Y0)
    Q1_2decoy = calc_Q1_2decoy(Y1_2decoy, mu_s)
    e1_2decoy = calc_e1_2decoy(mu_1, mu_2, Ti, Y0, e0, e_det, Y1_2decoy)
    e1_all.append(e1_2decoy)
    Q1_all.append(Q1_2decoy)
e1_all, Q1_all = np.array(e1_all), np.array(Q1_all)

QBER_MIMO = np.sum(e1_all * Q1_all) / np.sum(Q1_all)

print("Ti_all (simulated channel):", Ti_all_sim)
print("e1_all (per sub-channel):", e1_all)
print("Q1_all (per sub-channel):", Q1_all)
print("QBER_MIMO (paper formula, Eq. 20):", QBER_MIMO)

In [ ]:
# ================= Step 29: Compute SKR (Paper Formula, from Simulated Channel) =================

def H2(x):
    x = np.clip(x, 1e-12, 1-1e-12)
    return -x*np.log2(x) - (1-x)*np.log2(1-x)

def calc_SKR_i(Ti, mu_s, mu_1, mu_2, Y0, e0, e_det, q, g_val):
    Q_mus = calc_Q(mu_s, Ti, Y0)
    E_mus = calc_E(mu_s, Ti, Y0, e0, e_det)
    Y1_2decoy = calc_Y1_2decoy(mu_s, mu_1, mu_2, Ti, Y0)
    Q1_2decoy = calc_Q1_2decoy(Y1_2decoy, mu_s)
    e1_2decoy = calc_e1_2decoy(mu_1, mu_2, Ti, Y0, e0, e_det, Y1_2decoy)
    term1 = Q1_2decoy * (1 - H2(e1_2decoy))
    term2 = Q_mus * g_val * H2(E_mus)
    return q * (term1 - term2)

SKR_i_all = np.array([calc_SKR_i(Ti, mu_s, mu_1, mu_2, Y0, e0, e_det, q, g_val) for Ti in Ti_all_sim])
SKR_MIMO = np.sum(SKR_i_all)

print("SKR per sub-channel:", SKR_i_all)
print("SKR_MIMO (paper formula, Eq. 13/19):", SKR_MIMO)

In [ ]:
from sequence.kernel.timeline import Timeline
from sequence.topology.node import QKDNode
from sequence.components.optical_channel import ClassicalChannel
from sequence.components.detector import QSDetectorPolarization
from sequence.qkd.BB84 import BB84, pair_bb84_protocols


def run_bb84_mimo_fso_simulation(NT, distance_km, verbose=True):
    """Runs one full SeQUeNCe BB84 + MIMO-FSO + PNS-attack simulation
    for the given MIMO configuration (NT=NR) and distance. Combines Steps 7-29.
    """
    NR = NT   # square MIMO configuration
    distance_m = distance_km * 1000

    tl = Timeline(10e12)
    alice = QKDNode("Alice", tl, stack_size=0)
    bob = QKDNode("Bob", tl, stack_size=0)

    eve = EveEntity("Eve")

    qc_a2b = MIMOFSOChannel("qc_a2b", tl, distance=distance_m, NT=NT, NR=NR,
                             polarization_fidelity=1 - e_det, eve=eve)
    qc_a2b.set_ends(alice, bob.name)
    qc_b2a = MIMOFSOChannel("qc_b2a", tl, distance=distance_m, NT=NT, NR=NR,
                             polarization_fidelity=1 - e_det)
    qc_b2a.set_ends(bob, alice.name)

    cc_a2b = ClassicalChannel("cc_a2b", tl, distance=distance_m)
    cc_a2b.set_ends(alice, bob.name)
    cc_b2a = ClassicalChannel("cc_b2a", tl, distance=distance_m)
    cc_b2a.set_ends(bob, alice.name)

    alice_source = DecoyWCPSource("Alice.wcp_source", tl, mu_s=mu_s, mu_1=mu_1, mu_2=mu_2, probs=decoy_probs)
    alice.add_component(alice_source)
    alice_source.add_receiver(alice)
    alice.destination = "Bob"

    bob_qsdetector = QSDetectorPolarization("Bob.qsdetector_custom", tl)
    for d in bob_qsdetector.detectors:
        d.dark_count = Y0
    bob.add_component(bob_qsdetector)
    for d in bob_qsdetector.detectors:
        bob.add_component(d)
    bob.add_component(bob_qsdetector.splitter)
    bob.set_first_component(bob_qsdetector.name)

    bb84_alice = BB84(alice, "bb84_alice", "Alice.wcp_source", "Bob.qsdetector_custom")
    bb84_bob = BB84(bob, "bb84_bob", "Alice.wcp_source", "Bob.qsdetector_custom")
    alice.protocols.append(bb84_alice)
    bob.protocols.append(bb84_bob)

    pair_bb84_protocols(bb84_alice, bb84_bob)

    tl.init()

    if verbose:
        print(f"\n===== NT=NR={NT}, distance={distance_km} km =====")
        print("qc_a2b betas:", qc_a2b.betas)
        print("qc_a2b Ti_all:", qc_a2b.Ti_all)
        print("qc_a2b loss:", qc_a2b.loss)

    qc_delay = qc_a2b.delay
    cc_delay = cc_a2b.delay
    round_trip_ps = qc_delay + 3 * cc_delay
    MARGIN = 5
    light_time_target_s = (round_trip_ps * MARGIN) / 1e12
    mean_photon_num_needed = N_BITS / (alice_source.frequency * light_time_target_s)
    alice_source.mean_photon_num = mean_photon_num_needed
    run_time_ps = light_time_target_s * 1e12 + 3 * cc_delay + round_trip_ps

    bb84_alice.push(length=N_BITS, key_num=1, run_time=run_time_ps)

    tl.run()

    key_match = bb84_alice.key == bb84_bob.key
    error_rate = bb84_alice.error_rates[0] if bb84_alice.error_rates else None

    if verbose:
        print(f"Bob detector clicks: {bob_qsdetector.detectors[0].photon_counter}, {bob_qsdetector.detectors[1].photon_counter}")
        print(f"Eve stole photons at pulses: {eve.stored_photons}")
        print(f"Alice key: {bb84_alice.key} | Bob key: {bb84_bob.key} | Match: {key_match}")
        print(f"Simulated raw error rate: {error_rate}")

    Ti_all_sim = qc_a2b.Ti_all
    e1_all, Q1_all = [], []
    for Ti in Ti_all_sim:
        Y1_2decoy = calc_Y1_2decoy(mu_s, mu_1, mu_2, Ti, Y0)
        Q1_2decoy = calc_Q1_2decoy(Y1_2decoy, mu_s)
        e1_2decoy = calc_e1_2decoy(mu_1, mu_2, Ti, Y0, e0, e_det, Y1_2decoy)
        e1_all.append(e1_2decoy)
        Q1_all.append(Q1_2decoy)
    e1_all, Q1_all = np.array(e1_all), np.array(Q1_all)
    QBER_MIMO = np.sum(e1_all * Q1_all) / np.sum(Q1_all)

    SKR_i_all = np.array([calc_SKR_i(Ti, mu_s, mu_1, mu_2, Y0, e0, e_det, q, g_val) for Ti in Ti_all_sim])
    SKR_MIMO = np.sum(SKR_i_all)

    if verbose:
        print(f"QBER_MIMO (paper formula): {QBER_MIMO}")
        print(f"SKR_MIMO (paper formula): {SKR_MIMO}")

    return {
        "NT": NT, "distance_km": distance_km,
        "QBER_MIMO": QBER_MIMO, "SKR_MIMO": SKR_MIMO,
        "key_match": key_match, "simulated_error_rate": error_rate,
        "eve_stolen_count": len(eve.stored_photons),
    }

In [ ]:
# ================= Step 30: Loop Over MIMO_configs and distances_km_graph =================
results = []

for NT in MIMO_configs:
    for distance_km in distances_km_graph:
        is_first_run = (len(results) == 0)   # full verbosity only for the very first combination
        result = run_bb84_mimo_fso_simulation(NT, distance_km, verbose=is_first_run)
        results.append(result)
        print(f"[Summary] NT={NT} distance={distance_km}km -> QBER={result['QBER_MIMO']:.6f} "
              f"SKR={result['SKR_MIMO']:.6f} key_match={result['key_match']}")

print()
print(f"Completed {len(results)} combinations ({len(MIMO_configs)} MIMO configs x {len(distances_km_graph)} distances).")

In [ ]:
# ================= Step 31: Generate Performance Graphs =================
import plotly.graph_objects as go

QBER_by_NT = {NT: [] for NT in MIMO_configs}
SKR_by_NT = {NT: [] for NT in MIMO_configs}

for r in results:
    QBER_by_NT[r["NT"]].append(r["QBER_MIMO"])
    SKR_by_NT[r["NT"]].append(r["SKR_MIMO"])

# ---- Plot: QBER vs Distance ----
fig_qber = go.Figure()
for NT in MIMO_configs:
    fig_qber.add_trace(go.Scatter(x=distances_km_graph, y=QBER_by_NT[NT],
                                   mode="lines+markers", name=f"NT=NR={NT}"))
fig_qber.update_layout(title="QBER vs Distance (Bb84 One-Way)",
                        xaxis_title="Distance (km)", yaxis_title="QBER_MIMO",
                        template="plotly_white")
fig_qber.show()

# ---- Plot: SKR vs Distance ----
fig_skr = go.Figure()
for NT in MIMO_configs:
    fig_skr.add_trace(go.Scatter(x=distances_km_graph, y=SKR_by_NT[NT],
                                  mode="lines+markers", name=f"NT=NR={NT}"))
fig_skr.update_layout(title="SKR vs Distance (BB84 One-Way)",
                       xaxis_title="Distance (km)", yaxis_title="SKR_MIMO (bps)",
                       yaxis_type="log", template="plotly_white")
fig_skr.show()

In [ ]:
print(results[0].keys())

In [ ]:
# ================= ADD THIS AS A NEW CELL AT THE END OF SeQUeNCe_BB84_Simulator_7.ipynb =================
# Saves the BB84 sweep results (the `results` list) to a JSON file,
# now including sifted_key_length so it matches the LM05 export structure.

import json

bb84_export = [
    {
        "NT": r["NT"],
        "distance_km": int(r["distance_km"]),
        "QBER": float(r["QBER_MIMO"]),
        "SKR": float(r["SKR_MIMO"]),
        #"sifted_key_length": int(r["sifted_key_length"]),
        "key_match": bool(r["key_match"]),
    }
    for r in results
]

with open("bb84_results.json", "w") as f:
    json.dump(bb84_export, f, indent=2)

print(f"Saved {len(bb84_export)} BB84 results -> bb84_results.json")